In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [2]:
import joblib

project_root = Path("/Users/alexgonzalez/Documents/NBA-Prop-Predictor")  # or Path.cwd()
min_path = project_root / "src/models/saved_models/min_quantile_xgb_2026-01-02.joblib"
ppm_path = project_root / "src/models/saved_models/ppm_quantile_xgb_2025-12-31.joblib"
apm_path = project_root / "src/models/saved_models/apm_quantile_xgb_2025-12-31.joblib"
rpm_path = project_root / "src/models/saved_models/rpm_quantile_xgb_2025-12-31.joblib"

min_bundle = joblib.load(min_path)
min_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

ppm_bundle = joblib.load(ppm_path)
ppm_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

apm_bundle = joblib.load(apm_path)
apm_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

rpm_bundle = joblib.load(rpm_path)
rpm_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

### Helper Functions

In [3]:
from src.live import *

MARKET_RATE_COL = {
    'PTS': 'PTS_PER_MIN',
    'AST': 'AST_PER_MIN',
    'REB': 'REB_PER_MIN',
}

def get_rate_history(df, player, date, n_games=20, *, market=None, rate_col=None):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] < date)]
    pdf = pdf.sort_values(by='GAME_DATE', ascending=True)

    if rate_col is None:
        if market is None:
            raise ValueError("Provide either market=... or rate_col=...")
        rate_col = MARKET_RATE_COL.get(market, f"{market}_PER_MIN")

    if rate_col not in pdf.columns:
        raise KeyError(rate_col)

    rate_history = pdf[rate_col].dropna().tail(n_games)
    return rate_history

def grab_player_last_game(df, features, player, date):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] == date)]
    if pdf.empty:
        return f"No data found for {player} on {date}"
    return pdf[features]

def build_sim_row(player, min_arr, rate_arr, min_models, rate_models, rate_history, *, market='PTS'):
    """Build a row dict compatible with `run_pts_simulation` in `src/live.py`."""
    q10, q50, q90 = 'q_0.10', 'q_0.50', 'q_0.90'

    m10 = float(min_models[q10].predict(min_arr)[0])
    m50 = float(min_models[q50].predict(min_arr)[0])
    m90 = float(min_models[q90].predict(min_arr)[0])

    r10 = float(rate_models[q10].predict(rate_arr)[0])
    r50 = float(rate_models[q50].predict(rate_arr)[0])
    r90 = float(rate_models[q90].predict(rate_arr)[0])

    return {
        'PLAYER_NAME': player,
        'MARKET': market,
        'MIN_Q10': m10, 'MIN_Q50': m50, 'MIN_Q90': m90,
        'RATE_Q10': r10, 'RATE_Q50': r50, 'RATE_Q90': r90,
        'STAT_Q10': m10 * r10, 'STAT_Q50': m50 * r50, 'STAT_Q90': m90 * r90,
        'RATE_HISTORY': list(rate_history) if rate_history is not None else [],
    }


### Solo lookup

In [8]:
player = 'Shai Gilgeous-Alexander'
date = '2026-04-02'
last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)

min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)

min_pred = min_models['q_0.50'].predict(min_arr)
ppm_pred = ppm_models['q_0.50'].predict(ppm_arr)
ppm_pred_q90 = ppm_models['q_0.90'].predict(ppm_arr)
rate_history = get_rate_history(pts_df, player, '2026-04-02', rate_col='PTS_PER_MIN').to_list()

print(f"predicted MIN: {min_pred[0]:.2f}, predicted PPM: {ppm_pred[0]:.2f}, predicted PTS: {min_pred[0] * ppm_pred[0]:.2f}, predicted PTS q90: {min_pred[0] * ppm_pred_q90[0]:.2f}")
print(f"rate history: {rate_history}")
last_ppm

predicted MIN: 34.96, predicted PPM: 0.96, predicted PTS: 33.57, predicted PTS q90: 41.67
rate history: [0.6818827540486789, 0.8094152672465925, 0.9549071618037136, 1.016949152542373, 0.7194244604316546, 1.0661401776900297, 0.8962804361898123, 0.7330827067669172, 0.7714285714285715, 0.8985879332477534, 0.8955223880597014, 0.6018054162487462, 1.1230697239120262, 0.7822685788787485, 1.2572027239392354, 0.7573149741824441, 0.8928571428571428, 0.8479366873940078, 0.8148483476686282, 1.1595394736842106]


,UFGA_PER_MIN_X_OPP_DEF_RATING,CFGA_PER_MIN_X_OPP_DEF_RATING,PPM_SEASON_MEAN,TEAM_USG_RANK_L10,PTS_PER_MIN_X_OPP_PTS_ALLOWED,TS_PCT_X_USG_PCT,FT_PCT_season_avg,PTS_season_avg
147073,45.9,28.35,0.944005,1.0,106.66821,0.2176,0.89,31.62


### Week lookup 

In [9]:
date = '2025-12-25'  # game date for backtest
N_SIMS = 10_000
PROB_THRESHOLD = 0.58
STAKE = 100.0

np.random.seed(42)

backtest_df = pd.read_csv(
    '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_lines/NBA_US_20251225_135017.csv'
)
backtest_df = backtest_df[backtest_df['CATEGORY'] == 'player_points'].copy()

# Pivot so each (bookmaker, player, line) row has both Over + Under odds
lines_df = (
    backtest_df
    .pivot_table(
        index=['BOOKMAKER', 'NAME', 'LINE'],
        columns='OVER/UNDER',
        values='ODDS',
        aggfunc='first',
    )
    .reset_index()
    .rename(columns={'Over': 'odds_over', 'Under': 'odds_under'})
)

def american_to_profit(odds, stake=STAKE):
    odds = float(odds)
    if odds > 0:
        return stake * (odds / 100.0)
    return stake * (100.0 / abs(odds))

# Cache simulations per player so we don't redo them for every bookmaker
sim_cache = {}

def get_sims_for_player(player):
    if player in sim_cache:
        return sim_cache[player]

    pdf = pts_df[(pts_df['PLAYER_NAME'] == player) & (pts_df['GAME_DATE'] == date)]
    if pdf.empty:
        sim_cache[player] = None
        return None

    last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
    last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)
    if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_ppm, pd.DataFrame):
        sim_cache[player] = None
        return None

    min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
    ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)
    rate_history = get_rate_history(pts_df, player, date, n_games=20, market='PTS').tolist()
    sim_row = build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history, market='PTS')
    sims = run_pts_simulation(sim_row, n_sims=N_SIMS)

    payload = {
        'sims': sims,
        'sim_row': sim_row,
        'actual_pts': pdf['PTS'].values[0],
        'actual_min': pdf['MIN'].values[0],
        'player_pts_mean': pts_df[pts_df['PLAYER_NAME'] == player]['PTS'].mean(),
    }
    sim_cache[player] = payload
    return payload

res = []
missing_players = set()

for _, row in lines_df.iterrows():
    player = row['NAME']
    book = row['BOOKMAKER']
    line = float(row['LINE'])
    odds_over = row.get('odds_over')
    odds_under = row.get('odds_under')

    payload = get_sims_for_player(player)
    if payload is None:
        missing_players.add(player)
        continue

    sims = payload['sims']
    sim_row = payload['sim_row']
    actual_pts = payload['actual_pts']

    p_over = float(np.mean(sims > line))
    p_under = float(np.mean(sims < line))
    sim_mean = float(np.mean(sims))
    sim_p10, sim_p50, sim_p90 = np.percentile(sims, [10, 50, 90])

    pred = sim_row['STAT_Q50']
    side = 'Over' if p_over >= p_under else 'Under'
    p_side = p_over if side == 'Over' else p_under
    used_odds = odds_over if side == 'Over' else odds_under

    if pd.isna(used_odds):
        continue

    hit = (side == 'Over' and actual_pts > line) or (side == 'Under' and actual_pts < line)
    edge = abs(pred - line)
    recommended_bet = 1 if ((p_side >= PROB_THRESHOLD) and (edge > 2.5)) else 0
    profit = american_to_profit(used_odds) if hit else -STAKE

    res.append({
        'bookmaker': book,
        'player': player,
        'player_pts_mean': payload['player_pts_mean'],
        'pred_points': round(pred, 2),
        'sim_mean': round(sim_mean, 2),
        'sim_p10': round(float(sim_p10), 2),
        'sim_p50': round(float(sim_p50), 2),
        'sim_p90': round(float(sim_p90), 2),
        'side': side,
        'line': line,
        'odds_used': used_odds,
        'odds_over': odds_over,
        'odds_under': odds_under,
        'actual_points': actual_pts,
        'p_over': round(p_over, 3),
        'p_under': round(p_under, 3),
        'p_side': round(p_side, 3),
        'edge': round(edge, 2),
        'hit': int(hit),
        'stake': STAKE,
        'profit': round(profit, 2),
        'recommended_bet': recommended_bet,
    })

res = pd.DataFrame(res)

print('-' * 100)
print(f"Books: {res['bookmaker'].nunique()} | Players: {res['player'].nunique()} | Bets: {len(res)}")
print(f"Hit rate: {round(res['hit'].mean(), 3)}")
print(f"Total staked: ${len(res) * STAKE:.2f} | Profit: ${res['profit'].sum():.2f} "
      f"| ROI: {round(res['profit'].sum() / (len(res) * STAKE) * 100, 2)}%")

rec = res[res['recommended_bet'] == 1]
if len(rec):
    print('-' * 100)
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].mean(), 3)}")
    print(f"Staked: ${len(rec) * STAKE:.2f} | Profit: ${rec['profit'].sum():.2f} "
          f"| ROI: {round(rec['profit'].sum() / (len(rec) * STAKE) * 100, 2)}%")

if missing_players:
    print('-' * 100)
    print(f"Skipped {len(missing_players)} players with no data on {date}: "
          f"{sorted(missing_players)[:10]}{' ...' if len(missing_players) > 10 else ''}")

res.head(10)

----------------------------------------------------------------------------------------------------
Books: 6 | Players: 76 | Bets: 826
Hit rate: 0.556
Total staked: $82600.00 | Profit: $-2808.15 | ROI: -3.4%
----------------------------------------------------------------------------------------------------
Recommended bets (p_side >= 0.58): 204 | hit rate: 0.686
Staked: $20400.00 | Profit: $1610.92 | ROI: 7.9%
----------------------------------------------------------------------------------------------------
Skipped 1 players with no data on 2025-12-25: ['Jalen Pickett']


,bookmaker,player,player_pts_mean,pred_points,sim_mean,sim_p10,sim_p50,sim_p90,side,line,odds_used,odds_over,odds_under,actual_points,p_over,p_under,p_side,edge,hit,stake,profit,recommended_bet
0,BetMGM,Aaron Wiggins,8.471850,7.15,8.76,3.00,7.26,17.00,Under,7.5,-120,-110,-120,5,0.479,0.520,0.520,0.35,1,100.0,83.33,0
1,BetMGM,Al Horford,9.571749,6.78,7.06,1.85,6.74,12.89,Over,6.5,100,100,-135,14,0.528,0.472,0.528,0.28,1,100.0,100.00,0
2,BetMGM,Alex Caruso,7.266150,5.51,6.38,2.08,5.76,11.99,Under,6.5,-140,105,-140,12,0.409,0.591,0.591,0.99,0,100.0,-100.00,0
3,BetMGM,Alperen Sengun,17.029891,25.99,24.82,13.04,24.88,35.66,Over,21.5,-118,-118,-115,14,0.659,0.341,0.659,4.49,0,100.0,-100.00,1
4,BetMGM,Amen Thompson,14.427273,18.33,18.68,9.62,17.83,28.44,Over,17.5,-115,-115,-118,26,0.519,0.480,0.519,0.83,1,100.0,86.96,0
5,BetMGM,Anthony Davis,23.850467,22.83,24.75,12.75,24.94,37.28,Over,24.5,-110,-110,-118,3,0.520,0.480,0.520,1.67,0,100.0,-100.00,0
6,BetMGM,Anthony Edwards,24.823293,29.06,29.61,15.50,29.02,43.84,Under,30.5,-120,-110,-120,44,0.449,0.551,0.551,1.44,0,100.0,-100.00,0
7,BetMGM,Austin Reaves,15.935657,21.52,24.93,11.32,24.56,39.24,Over,19.5,-105,-105,-125,12,0.649,0.351,0.649,2.02,0,100.0,-100.00,0
8,BetMGM,Brandin Podziemski,11.625000,11.47,13.07,5.95,12.67,20.78,Over,10.5,-118,-118,-110,13,0.654,0.346,0.654,0.97,1,100.0,84.75,0
9,BetMGM,Bruce Brown,9.721277,8.77,9.89,2.18,10.08,16.41,Over,8.5,-105,-105,-125,7,0.628,0.372,0.628,0.27,0,100.0,-100.00,0


In [6]:
def summarize_by_book(df, label):
    if df.empty:
        print(f"No bets for: {label}")
        return None
    out = (
        df.groupby('bookmaker')
          .agg(
              bets=('hit', 'size'),
              hits=('hit', 'sum'),
              hit_rate=('hit', 'mean'),
              staked=('stake', 'sum'),
              profit=('profit', 'sum'),
          )
          .assign(roi_pct=lambda x: (x['profit'] / x['staked'] * 100).round(2))
          .sort_values('roi_pct', ascending=False)
    )
    out['hit_rate'] = out['hit_rate'].round(3)
    out['profit'] = out['profit'].round(2)
    out['staked'] = out['staked'].round(2)
    print(f"\n=== ROI per bookmaker — {label} ===")
    print(out.to_string())
    return out

book_all = summarize_by_book(res, 'all bets')
book_rec = summarize_by_book(res[res['recommended_bet'] == 1],
                             f'recommended bets (p_side >= {PROB_THRESHOLD})')


=== ROI per bookmaker — all bets ===
              bets  hits  hit_rate   staked   profit  roi_pct
bookmaker                                                    
BetOnline.ag    57    32     0.561   5700.0   254.65     4.47
DraftKings      58    32     0.552   5800.0   197.39     3.40
BetMGM          61    33     0.541   6100.0    41.83     0.69
FanDuel         56    30     0.536   5600.0     0.81     0.01
BetRivers      126    66     0.524  12600.0  -574.66    -4.56
Bovada         468   266     0.568  46800.0 -2728.17    -5.83

=== ROI per bookmaker — recommended bets (p_side >= 0.58) ===
              bets  hits  hit_rate   staked   profit  roi_pct
bookmaker                                                    
DraftKings       9     7     0.778    900.0   371.29    41.25
BetOnline.ag     8     6     0.750    800.0   284.68    35.58
Bovada         149   107     0.718  14900.0  1165.80     7.82
BetMGM           9     5     0.556    900.0    16.52     1.84
BetRivers       21    11     0.

In [10]:
date = '2025-12-25'  # game date for backtest
N_SIMS = 10_000
PROB_THRESHOLD = 0.58
STAKE = 100.0

np.random.seed(42)

backtest_df = pd.read_csv(
    '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_lines/NBA_US_20251225_135017.csv'
)
backtest_df = backtest_df[backtest_df['CATEGORY'] == 'player_assists'].copy()

# Pivot so each (bookmaker, player, line) row has both Over + Under odds
lines_df = (
    backtest_df
    .pivot_table(
        index=['BOOKMAKER', 'NAME', 'LINE'],
        columns='OVER/UNDER',
        values='ODDS',
        aggfunc='first',
    )
    .reset_index()
    .rename(columns={'Over': 'odds_over', 'Under': 'odds_under'})
)

def american_to_profit(odds, stake=STAKE):
    odds = float(odds)
    if odds > 0:
        return stake * (odds / 100.0)
    return stake * (100.0 / abs(odds))

# Cache simulations per player so we don't redo them for every bookmaker
sim_cache = {}

def get_sims_for_player(player):
    if player in sim_cache:
        return sim_cache[player]

    pdf = ast_df[(ast_df['PLAYER_NAME'] == player) & (ast_df['GAME_DATE'] == date)]
    if pdf.empty:
        sim_cache[player] = None
        return None

    last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
    last_apm = grab_player_last_game(ast_df, apm_feature_names, player, date)
    if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_apm, pd.DataFrame):
        sim_cache[player] = None
        return None

    min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
    apm_arr = np.asarray(last_apm, dtype=float).reshape(1, -1)
    rate_history = get_rate_history(ast_df, player, date, n_games=20, market='AST').tolist()
    sim_row = build_sim_row(player, min_arr, apm_arr, min_models, apm_models, rate_history, market='AST')
    sims = run_pts_simulation(sim_row, n_sims=N_SIMS)

    payload = {
        'sims': sims,
        'sim_row': sim_row,
        'actual_ast': pdf['AST'].values[0],
        'actual_min': pdf['MIN'].values[0],
        'player_ast_mean': ast_df[ast_df['PLAYER_NAME'] == player]['AST'].mean(),
    }
    sim_cache[player] = payload
    return payload

res = []
missing_players = set()

for _, row in lines_df.iterrows():
    player = row['NAME']
    book = row['BOOKMAKER']
    line = float(row['LINE'])
    odds_over = row.get('odds_over')
    odds_under = row.get('odds_under')

    payload = get_sims_for_player(player)
    if payload is None:
        missing_players.add(player)
        continue

    sims = payload['sims']
    sim_row = payload['sim_row']
    actual_ast = payload['actual_ast']

    p_over = float(np.mean(sims > line))
    p_under = float(np.mean(sims < line))
    sim_mean = float(np.mean(sims))
    sim_p10, sim_p50, sim_p90 = np.percentile(sims, [10, 50, 90])

    pred = sim_row['STAT_Q50']
    side = 'Over' if p_over >= p_under else 'Under'
    p_side = p_over if side == 'Over' else p_under
    used_odds = odds_over if side == 'Over' else odds_under

    if pd.isna(used_odds):
        continue

    hit = (side == 'Over' and actual_ast > line) or (side == 'Under' and actual_ast < line)
    edge = abs(pred - line)
    recommended_bet = 1 if ((p_side >= PROB_THRESHOLD) and (edge > 1.5)) else 0
    profit = american_to_profit(used_odds) if hit else -STAKE

    res.append({
        'bookmaker': book,
        'player': player,
        'player_ast_mean': payload['player_ast_mean'],
        'pred_assists': round(pred, 2),
        'sim_mean': round(sim_mean, 2),
        'sim_p10': round(float(sim_p10), 2),
        'sim_p50': round(float(sim_p50), 2),
        'sim_p90': round(float(sim_p90), 2),
        'side': side,
        'line': line,
        'odds_used': used_odds,
        'odds_over': odds_over,
        'odds_under': odds_under,
        'actual_assists': actual_ast,
        'p_over': round(p_over, 3),
        'p_under': round(p_under, 3),
        'p_side': round(p_side, 3),
        'edge': round(edge, 2),
        'hit': int(hit),
        'stake': STAKE,
        'profit': round(profit, 2),
        'recommended_bet': recommended_bet,
    })

res = pd.DataFrame(res)

print('-' * 100)
print(f"Books: {res['bookmaker'].nunique()} | Players: {res['player'].nunique()} | Bets: {len(res)}")
print(f"Hit rate: {round(res['hit'].mean(), 3)}")
print(f"Total staked: ${len(res) * STAKE:.2f} | Profit: ${res['profit'].sum():.2f} "
      f"| ROI: {round(res['profit'].sum() / (len(res) * STAKE) * 100, 2)}%")

rec = res[res['recommended_bet'] == 1]
if len(rec):
    print('-' * 100)
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].mean(), 3)}")
    print(f"Staked: ${len(rec) * STAKE:.2f} | Profit: ${rec['profit'].sum():.2f} "
          f"| ROI: {round(rec['profit'].sum() / (len(rec) * STAKE) * 100, 2)}%")

if missing_players:
    print('-' * 100)
    print(f"Skipped {len(missing_players)} players with no data on {date}: "
          f"{sorted(missing_players)[:10]}{' ...' if len(missing_players) > 10 else ''}")

res.head(20)

----------------------------------------------------------------------------------------------------
Books: 6 | Players: 67 | Bets: 278
Hit rate: 0.471
Total staked: $27800.00 | Profit: $-5032.45 | ROI: -18.1%
----------------------------------------------------------------------------------------------------
Recommended bets (p_side >= 0.58): 5 | hit rate: 0.4
Staked: $500.00 | Profit: $-203.89 | ROI: -40.78%
----------------------------------------------------------------------------------------------------
Skipped 1 players with no data on 2025-12-25: ['Jalen Pickett']


,bookmaker,player,player_ast_mean,pred_assists,sim_mean,sim_p10,sim_p50,sim_p90,side,line,odds_used,odds_over,odds_under,actual_assists,p_over,p_under,p_side,edge,hit,stake,profit,recommended_bet
0,BetMGM,Aaron Wiggins,1.367292,1.14,1.49,0.00,1.28,3.37,Over,0.5,-225,-225,170,0,0.734,0.266,0.734,0.64,0,100.0,-100.00,0
1,BetMGM,Al Horford,2.818386,2.46,2.90,0.32,2.58,5.91,Over,1.5,-110,-110,-118,2,0.684,0.316,0.684,0.96,1,100.0,90.91,0
2,BetMGM,Alex Caruso,2.850129,2.14,2.46,0.62,2.25,4.79,Under,2.5,-145,105,-145,0,0.448,0.552,0.552,0.36,1,100.0,68.97,0
3,BetMGM,Alperen Sengun,4.508152,6.90,6.90,3.06,6.59,10.92,Over,5.5,-155,-155,115,4,0.642,0.358,0.642,1.40,0,100.0,-100.00,0
4,BetMGM,Amen Thompson,4.050000,5.91,5.90,2.75,5.80,9.03,Over,5.5,105,105,-145,5,0.553,0.447,0.553,0.41,0,100.0,-100.00,0
5,BetMGM,Anthony Davis,3.109034,2.96,3.41,1.42,2.91,6.22,Over,2.5,-145,-145,105,0,0.594,0.406,0.594,0.46,0,100.0,-100.00,0
6,BetMGM,Anthony Edwards,4.240964,4.34,4.15,1.81,3.89,6.93,Under,4.5,-135,100,-135,3,0.396,0.604,0.604,0.16,1,100.0,74.07,0
7,BetMGM,Austin Reaves,4.445040,5.37,5.67,2.60,5.08,9.60,Over,3.5,-145,-145,110,1,0.723,0.277,0.723,1.87,0,100.0,-100.00,1
8,BetMGM,Brandin Podziemski,3.594828,3.63,4.13,1.55,4.21,6.67,Over,3.5,120,120,-160,4,0.617,0.383,0.617,0.13,1,100.0,120.00,0
9,BetMGM,Bruce Brown,2.340426,2.15,2.50,0.22,2.01,5.18,Over,1.5,-175,-175,130,3,0.595,0.405,0.595,0.65,1,100.0,57.14,0


In [11]:
book_all = summarize_by_book(res, 'all bets')
book_rec = summarize_by_book(res[res['recommended_bet'] == 1],
                             f'recommended bets (p_side >= {PROB_THRESHOLD})')


=== ROI per bookmaker — all bets ===
              bets  hits  hit_rate  staked   profit  roi_pct
bookmaker                                                   
FanDuel         30    15     0.500  3000.0  -323.80   -10.79
Bovada          86    44     0.512  8600.0 -1438.42   -16.73
BetMGM          61    28     0.459  6100.0 -1137.91   -18.65
BetOnline.ag    37    17     0.459  3700.0  -720.64   -19.48
DraftKings      36    15     0.417  3600.0  -772.62   -21.46
BetRivers       28    12     0.429  2800.0  -639.06   -22.82

=== ROI per bookmaker — recommended bets (p_side >= 0.58) ===
           bets  hits  hit_rate  staked  profit  roi_pct
bookmaker                                               
Bovada        3     2     0.667   300.0   -3.89     -1.3
BetMGM        1     0     0.000   100.0 -100.00   -100.0
BetRivers     1     0     0.000   100.0 -100.00   -100.0


In [12]:
date = '2025-12-25'  # game date for backtest
N_SIMS = 10_000
PROB_THRESHOLD = 0.58
STAKE = 100.0

np.random.seed(42)

backtest_df = pd.read_csv(
    '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_lines/NBA_US_20251225_135017.csv'
)
backtest_df = backtest_df[backtest_df['CATEGORY'] == 'player_rebounds'].copy()

lines_df = (
    backtest_df
    .pivot_table(
        index=['BOOKMAKER', 'NAME', 'LINE'],
        columns='OVER/UNDER',
        values='ODDS',
        aggfunc='first',
    )
    .reset_index()
    .rename(columns={'Over': 'odds_over', 'Under': 'odds_under'})
)

def american_to_profit(odds, stake=STAKE):
    odds = float(odds)
    if odds > 0:
        return stake * (odds / 100.0)
    return stake * (100.0 / abs(odds))

sim_cache = {}

def get_sims_for_player(player):
    if player in sim_cache:
        return sim_cache[player]

    pdf = reb_df[(reb_df['PLAYER_NAME'] == player) & (reb_df['GAME_DATE'] == date)]
    if pdf.empty:
        sim_cache[player] = None
        return None

    last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
    last_rpm = grab_player_last_game(reb_df, rpm_feature_names, player, date)
    if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_rpm, pd.DataFrame):
        sim_cache[player] = None
        return None

    min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
    rpm_arr = np.asarray(last_rpm, dtype=float).reshape(1, -1)

    rate_history = get_rate_history(reb_df, player, date, n_games=20, market='REB').tolist()
    sim_row = build_sim_row(player, min_arr, rpm_arr, min_models, rpm_models, rate_history, market='REB')

    # Note: name is historical; it simulates MIN * RATE using your row fields.
    sims = run_pts_simulation(sim_row, n_sims=N_SIMS)

    payload = {
        'sims': sims,
        'sim_row': sim_row,
        'actual_reb': float(pdf['REB'].values[0]),
        'actual_min': float(pdf['MIN'].values[0]),
        'player_reb_mean': reb_df[reb_df['PLAYER_NAME'] == player]['REB'].mean(),
    }
    sim_cache[player] = payload
    return payload

res = []
missing_players = set()

for _, row in lines_df.iterrows():
    player = row['NAME']
    book = row['BOOKMAKER']
    line = float(row['LINE'])
    odds_over = row.get('odds_over')
    odds_under = row.get('odds_under')

    payload = get_sims_for_player(player)
    if payload is None:
        missing_players.add(player)
        continue

    sims = payload['sims']
    sim_row = payload['sim_row']
    actual_reb = payload['actual_reb']

    p_over = float(np.mean(sims > line))
    p_under = float(np.mean(sims < line))
    sim_mean = float(np.mean(sims))
    sim_p10, sim_p50, sim_p90 = np.percentile(sims, [10, 50, 90])

    pred = sim_row['STAT_Q50']
    side = 'Over' if p_over >= p_under else 'Under'
    p_side = p_over if side == 'Over' else p_under
    used_odds = odds_over if side == 'Over' else odds_under

    if pd.isna(used_odds):
        continue

    hit = (side == 'Over' and actual_reb > line) or (side == 'Under' and actual_reb < line)
    edge = abs(pred - line)

    # tweak threshold for rebounds vs assists/points as you like
    recommended_bet = 1 if ((p_side >= PROB_THRESHOLD) and (edge > 1.5)) else 0

    profit = american_to_profit(used_odds) if hit else -STAKE

    res.append({
        'bookmaker': book,
        'player': player,
        'player_reb_mean': payload['player_reb_mean'],
        'pred_rebounds': round(pred, 2),
        'sim_mean': round(sim_mean, 2),
        'sim_p10': round(float(sim_p10), 2),
        'sim_p50': round(float(sim_p50), 2),
        'sim_p90': round(float(sim_p90), 2),
        'side': side,
        'line': line,
        'odds_used': used_odds,
        'odds_over': odds_over,
        'odds_under': odds_under,
        'actual_rebounds': actual_reb,
        'p_over': round(p_over, 3),
        'p_under': round(p_under, 3),
        'p_side': round(p_side, 3),
        'edge': round(edge, 2),
        'hit': int(hit),
        'stake': STAKE,
        'profit': round(profit, 2),
        'recommended_bet': recommended_bet,
    })

res = pd.DataFrame(res)

print('-' * 100)
print(f"Books: {res['bookmaker'].nunique()} | Players: {res['player'].nunique()} | Bets: {len(res)}")
print(f"Hit rate: {round(res['hit'].mean(), 3)}")
print(f"Total staked: ${len(res) * STAKE:.2f} | Profit: ${res['profit'].sum():.2f} "
      f"| ROI: {round(res['profit'].sum() / (len(res) * STAKE) * 100, 2)}%")

rec = res[res['recommended_bet'] == 1]
if len(rec):
    print('-' * 100)
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].mean(), 3)}")
    print(f"Staked: ${len(rec) * STAKE:.2f} | Profit: ${rec['profit'].sum():.2f} "
          f"| ROI: {round(rec['profit'].sum() / (len(rec) * STAKE) * 100, 2)}%")

if missing_players:
    print('-' * 100)
    print(f"Skipped {len(missing_players)} players with no data on {date}: "
          f"{sorted(missing_players)[:10]}{' ...' if len(missing_players) > 10 else ''}")

res.head(20)

----------------------------------------------------------------------------------------------------
Books: 6 | Players: 73 | Bets: 450
Hit rate: 0.509
Total staked: $45000.00 | Profit: $-4434.17 | ROI: -9.85%
----------------------------------------------------------------------------------------------------
Recommended bets (p_side >= 0.58): 49 | hit rate: 0.653
Staked: $4900.00 | Profit: $78.59 | ROI: 1.6%
----------------------------------------------------------------------------------------------------
Skipped 2 players with no data on 2025-12-25: ['Jalen Pickett', 'Jaxson Hayes']


,bookmaker,player,player_reb_mean,pred_rebounds,sim_mean,sim_p10,sim_p50,sim_p90,side,line,odds_used,odds_over,odds_under,actual_rebounds,p_over,p_under,p_side,edge,hit,stake,profit,recommended_bet
0,BetMGM,Aaron Wiggins,3.093834,2.28,2.52,0.28,2.41,4.58,Under,2.5,-210,155,-210,1.0,0.475,0.525,0.525,0.22,1,100.0,47.62,0
1,BetMGM,Al Horford,6.775785,4.49,5.28,2.23,4.99,8.83,Over,4.5,-110,-110,-120,4.0,0.570,0.430,0.570,0.01,0,100.0,-100.00,0
2,BetMGM,Alex Caruso,3.113695,3.10,4.50,1.41,3.80,8.78,Over,3.5,120,120,-160,1.0,0.548,0.452,0.548,0.40,0,100.0,-100.00,0
3,BetMGM,Alperen Sengun,8.706522,9.19,9.80,4.97,9.19,15.68,Over,8.5,120,120,-160,12.0,0.558,0.442,0.558,0.69,1,100.0,120.00,0
4,BetMGM,Amen Thompson,7.536364,7.25,7.98,4.10,7.81,12.18,Over,6.5,-130,-130,-105,7.0,0.654,0.346,0.654,0.75,1,100.0,76.92,0
5,BetMGM,Anthony Davis,11.573209,11.99,13.22,7.41,12.72,19.62,Over,12.5,100,100,-130,3.0,0.517,0.482,0.517,0.51,0,100.0,-100.00,0
6,BetMGM,Anthony Edwards,5.365462,4.92,5.60,2.05,5.10,9.84,Under,5.5,-135,100,-135,6.0,0.457,0.543,0.543,0.58,0,100.0,-100.00,0
7,BetMGM,Austin Reaves,4.008043,4.48,5.09,2.71,4.50,8.46,Over,3.5,-105,-105,-125,1.0,0.729,0.271,0.729,0.98,0,100.0,-100.00,0
8,BetMGM,Brandin Podziemski,5.306034,3.84,4.43,1.95,4.00,7.80,Under,4.5,-160,120,-160,8.0,0.409,0.591,0.591,0.66,0,100.0,-100.00,0
9,BetMGM,Bruce Brown,4.385106,4.46,5.41,1.57,5.07,9.85,Over,4.5,-120,-120,-110,3.0,0.578,0.422,0.578,0.04,0,100.0,-100.00,0


In [13]:
book_all = summarize_by_book(res, 'all bets')
book_rec = summarize_by_book(res[res['recommended_bet'] == 1],
                             f'recommended bets (p_side >= {PROB_THRESHOLD})')


=== ROI per bookmaker — all bets ===
              bets  hits  hit_rate   staked   profit  roi_pct
bookmaker                                                    
Bovada         176    97     0.551  17600.0 -1206.77    -6.86
FanDuel         51    26     0.510   5100.0  -368.42    -7.22
DraftKings      54    27     0.500   5400.0  -441.39    -8.17
BetOnline.ag    60    29     0.483   6000.0  -706.06   -11.77
BetMGM          61    28     0.459   6100.0  -868.03   -14.23
BetRivers       48    22     0.458   4800.0  -843.50   -17.57

=== ROI per bookmaker — recommended bets (p_side >= 0.58) ===
              bets  hits  hit_rate  staked  profit  roi_pct
bookmaker                                                  
BetMGM           4     3     0.750   400.0  113.93    28.48
BetRivers        3     2     0.667   300.0   44.19    14.73
Bovada          30    21     0.700  3000.0   76.41     2.55
DraftKings       4     2     0.500   400.0  -43.31   -10.83
FanDuel          4     2     0.500   400.0 